In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model

!pip install trl
from trl import SFTTrainer

# Install bitsandbytes for 4-bit quantization
!pip install -U bitsandbytes>=0.46.1

# 1. Configuration & NLP Task Setup
# We are using a Llama-3 or Mistral-style model architecture (via tiny-llama for speed)
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
dataset_name = "samsum" # NLP Dataset for dialogue summarization

# 2. Quantization (Essential for fine-tuning on a single GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 3. Load and Format the NLP Dataset
dataset = load_dataset(dataset_name, split="train[:500]")

def format_instruction(sample):
    return f"### Instruction: Summarize the following conversation.\n### Input: {sample['dialogue']}\n### Response: {sample['summary']}"

# 4. LoRA Configuration (Fine-tuning specific layers)
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # Target the attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# 5. Training Setup
training_args = TrainingArguments(
    output_dir="./nlp-sft-summary",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    save_steps=50,
    logging_steps=10,
    fp16=True, # Use Mixed Precision
    push_to_hub=False,
)

# 6. The SFT Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=format_instruction, # Applies our NLP template
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

# 7. Execute Training
trainer.train()

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`